In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

# Encode Categorical_Features

In [4]:
# Define the categorical features
categorical_features = ["Material"]

le = LabelEncoder()

for feature in categorical_features:
    x_train[feature] = le.fit_transform(x_train[feature])
    x_dev[feature] = le.transform(x_dev[feature])  

# Add Physical Column

In [5]:
def compute_physical_calc(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = (np.pi/4) * (4 * np.sqrt(t))**2 * (0.8 * 365) 
     return np.round(f_pull, 1) 

x_train['Physical_Calc'] = compute_physical_calc(x_train) 
x_dev['Physical_Calc'] = compute_physical_calc(x_dev)

# Fit Model

In [6]:
# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor.fit(x_train, y_train)

# Predict on the test set
predictions = regressor.predict(x_dev)

# Check Validation Data

In [7]:
# Convert to numpy arrays (if not already)
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(predictions).ravel()

# Sample index for plotting
sample_idx = np.arange(len(true_vals))

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sample_idx, y=true_vals, mode="markers",
    name="Original Values", marker=dict(color="red", size=6)
))

fig.add_trace(go.Scatter(
    x=sample_idx, y=pred_vals, mode="markers",
    name="Predicted Values", marker=dict(color="blue", size=6)
))

# Connecting lines (one per sample) 
for i in range(len(sample_idx)): 
    fig.add_trace(go.Scatter( 
        x=[sample_idx[i], sample_idx[i]], 
        y=[true_vals[i], pred_vals[i]], 
        mode="lines", 
        line=dict(color="gray", width=1), 
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN)",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.write_html("Graphs/TabPFN_Evaltrue.html")
fig.show()


# Check Validation Loss and R2

In [8]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(mean_squared_error(y_dev, predictions))
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  156.12
RMSE: 263.47
R2: 0.69
